# 11 · Raster change, clouds and missing evidence

**Question:** Can a model distinguish vegetation change from missing pixels?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Compare a map interpretation with a paired-pixel calculation, then quantify which claims the evidence supports.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## Predict before you run · 25 minutes
**Learn:** align two raster dates, preserve nodata, compute area from pixel size, and audit a visual claim.
Imagine a cloudy second image. Would treating cloud pixels as zero exaggerate vegetation loss?
These are invented vegetation-index grids, not satellite imagery. Both grids share a local metric frame,
extent, resolution and pixel alignment. Values are dimensionless; no sensor calibration is implied.
Change `loss_threshold` and predict whether the detected area will grow or shrink.


In [ ]:
from experiments import paired_change, grade_answer
size=24; pixel_m=25.; loss_threshold=.20
row,col=np.mgrid[:size,:size]
before=.55+.12*np.sin(col/5)*np.cos(row/6)
after=before.copy(); after[5:13,13:21]-=.30
before[0:3,0:8]=np.nan
after[8:11,10:18]=np.nan  # cloud overlaps some true change
result=paired_change(before,after,pixel_m,loss_threshold)
assert result['valid_pixels']==size*size-48
assert result['loss_pixels']<=result['valid_pixels']
assert result['loss_area_m2']==result['loss_pixels']*pixel_m**2
if loss_threshold==.20:
    assert result['loss_pixels']==49  # 64 changed pixels, 15 hidden by cloud
print('Paired valid pixels:',result['valid_pixels'],'of',size*size)
print('Observed loss area (m²):',result['loss_area_m2'])
print('Cloud-hidden change remains unknown to an observer.')


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,4))
cmap=plt.get_cmap('YlGn').copy(); cmap.set_bad('#bdbdbd')
extent=[0,size*pixel_m,0,size*pixel_m]
for ax,field,title in zip(axes[:2],[before,after],['Earlier index','Later index']):
    image=ax.imshow(field,origin='lower',extent=extent,cmap=cmap,vmin=0,vmax=1)
    ax.set(title=title,xlabel='East (m)',ylabel='North (m)')
fig.colorbar(image,ax=list(axes[:2]),label='Synthetic vegetation index',shrink=.75)
delta_cmap=plt.get_cmap('RdBu').copy(); delta_cmap.set_bad('#bdbdbd')
image=axes[2].imshow(result['delta'],origin='lower',extent=extent,cmap=delta_cmap,vmin=-.35,vmax=.35)
axes[2].set(title='Later minus earlier; gray = unknown',xlabel='East (m)',ylabel='North (m)')
fig.colorbar(image,ax=axes[2],label='Index change',shrink=.75)
Path('exports').mkdir(exist_ok=True)
fig.savefig('exports/11_change.png',bbox_inches='tight'); plt.show()
naive_loss=int(((np.nan_to_num(after)-np.nan_to_num(before)) < -loss_threshold).sum())
print('Incorrect zero-fill loss pixels:',naive_loss,'| paired-valid loss pixels:',result['loss_pixels'])
assert naive_loss >= result['loss_pixels']


## Export only the evidence
Send `11_prompt.json` and optionally `11_change.png` in a fresh Astra conversation.
The JSON provides an accessible numerical alternative to the image: rows run south to north,
columns west to east, and `null` means unobserved. Do not send the notebook or the answer key.
For an image-only discussion, ask where the change occurs and why gray pixels cannot establish loss;
do not claim exact pixel-count accuracy from that image. Score the numerical trial below separately.


In [ ]:
def serial_grid(grid):
    return [[float(v) if np.isfinite(v) else None for v in line] for line in grid]
prompt=dict(task='Compare only pixels valid on both dates. Loss is later minus earlier strictly below negative loss_threshold. Return ONLY JSON with valid_pixels, loss_pixels, loss_area_m2. Do not infer values under clouds.',
            synthetic=True,before=serial_grid(before),after=serial_grid(after),pixel_size_m=pixel_m,
            loss_threshold=loss_threshold,nodata=None,rows='south to north',columns='west to east')
export_json('11_prompt.json',prompt)
expected={key:result[key] for key in ['valid_pixels','loss_pixels','loss_area_m2']}


## Capture, score and discuss
The default below is a **deliberately wrong teaching fixture**, not a model response.
Replace the text with the exact response and fill in provenance after an actual trial.
Keep a copy of the original prompt. Explain why observed area is not an estimate of total hidden change.
**Try:** shift the later grid one pixel. Equal shapes no longer establish geographic alignment.


In [ ]:
response_source='FIXTURE - intentionally wrong zero-fill answer'
model_name='not run'; trial_date='not run'; interface='fixture'
response_text=json.dumps(dict(valid_pixels=size*size,loss_pixels=naive_loss,loss_area_m2=naive_loss*pixel_m**2))
score=grade_answer(response_text,expected,{'loss_area_m2':.01})
display(score)
export_json('11_trial.json',dict(response_source=response_source,model=model_name,date=trial_date,
            interface=interface,prompt=prompt,response_text=response_text,score=score,
            tolerance_m2=.01,reference=expected))


## Extended investigation: Separate misregistration from an actual changed patch

Translate a textured synthetic raster by one pixel and search a small set of corrections using median absolute residual. A separate injected change patch remains after alignment. Search candidates use a central comparison window, jointly valid pixels for each shift and explicit nodata padding; no pixels wrap around the image. This texture, shift and changed patch are controlled synthetic experiments, not measured environmental change. Compare their assumptions with the real Sentinel evidence in notebook 04.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Separate misregistration from an actual changed patch', ['Two images + nodata', 'Shift search without wrapping', 'Alignment residual maps', 'Registration needs stable texture'])
plt.show()


In [ ]:
from extension_tools import shift_grid
from experiments import paired_change
rng=np.random.default_rng(54); ex_reference=rng.uniform(.2,.8,(32,32))
ex_changed=ex_reference.copy(); ex_changed[12:18,15:22]-=.25
ex_observed=shift_grid(ex_changed,1,1); ex_observed[3:7,20:25]=np.nan
ex_candidates=[]
for dy in range(-2,3):
    for dx in range(-2,3):
        shifted=shift_grid(ex_observed,dy,dx); residual=(shifted-ex_reference)[4:-4,4:-4]
        ex_candidates.append((float(np.nanmedian(abs(residual))),dy,dx))
ex_score,ex_dy,ex_dx=min(ex_candidates); assert (ex_dy,ex_dx)==(-1,-1)
ex_aligned=shift_grid(ex_observed,ex_dy,ex_dx)
ex_raw=paired_change(ex_reference,ex_observed,20,.2); ex_fixed=paired_change(ex_reference,ex_aligned,20,.2)
assert ex_fixed['loss_pixels']==42
assert paired_change([[np.nan]],[[np.nan]],20,.2)['mean_change'] is None
assert paired_change([[.8,.5],[.4,.3]],[[.4,.5],[.4,.3]],10,.2)['loss_area_m2']==100
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,field,title in zip(axes,[ex_raw['delta'],ex_fixed['delta'],ex_fixed['loss']],['Before registration: false change','After alignment: changed patch remains','Detected loss on valid paired pixels']):
    image=ax.imshow(field,origin='lower',extent=[0,640,0,640],cmap='coolwarm',vmin=-.5 if field.dtype!=bool else 0,vmax=.5 if field.dtype!=bool else 1)
    ax.set(title=title,xlabel='Local east (m)',ylabel='Local north (m)'); fig.colorbar(image,ax=ax)
fig.tight_layout(); extension_figure=fig
extension_evidence=dict(search=[dict(median_abs_residual=s,dy=dy,dx=dx) for s,dy,dx in ex_candidates],aligned_loss_pixels=ex_fixed['loss_pixels'],pixel_m=20,nodata='masked, never zero-filled')
extension_expected=dict(correction_dy=-1,correction_dx=-1,loss_area_m2=16800)
extension_task='Select the shift minimizing median absolute residual and compute aligned loss area. Return correction_dy, correction_dx and loss_area_m2.'


In [ ]:
extension_prompt,extension_reference=export_investigation('11',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 11_extension_prompt.json and optionally 11_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `11_extension_prompt.json` and `11_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('11_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
